# M2 - Dimension 1: Metrica clasica (exact-match por spans)


Este notebook implementa la **Dimension 1** del harness de evaluacion de M2 para el
modelo **Clinical BERT** (`PlanTL-GOB-ES/roberta-base-biomedical-clinical-es`) fine-tuneado con LoRA en M1.

Reutiliza **sin modificar** el criterio de `micro_prf1_by_doc` que ya se uso como
`metric_for_best_model` durante el fine-tuning, para mantener trazabilidad entre
el F1 reportado en M1 y esta evaluacion de M2.

**Modelo:** cargado desde Google Drive (`M1/saved_models/clinical_bert-distemist-lora`),
subido con la celda 38 de `03_finetuning.ipynb`.

**Gold set:** 75 ejemplos de `raw_splits["test"]` con >= 5 entidades, generados en
`01_dataset_preparation.ipynb` y guardados como JSONL en Drive.

**Output:** `resultado_dimension1.json`, consumido por el harness.

---
## Seccion 0 - Setup y reproducibilidad

In [1]:
import json
import re
import random
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from peft import PeftModel

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("Seeds fijadas. Notebook determinista.")


Seeds fijadas. Notebook determinista.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import transformers, peft
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)


Mounted at /content/drive
torch       : 2.11.0+cu128
transformers: 5.16.1
peft        : 0.20.0


---
## Seccion 1 - Rutas del proyecto

In [3]:
# Raiz del proyecto en Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")

# Adaptador LoRA de Clinical BERT -- cargado desde Google Drive
# Contiene: adapter_config.json, adapter_model.safetensors, tokenizer.json
MODEL_DIR = PROJECT_ROOT / "M1" / "saved_models" / "clinical_bert-distemist-lora"

# Gold set generado en 01_dataset_preparation.ipynb
# Formato JSONL: {"input": str, "esperado": list[str], "criterio": str}
GOLD_SET_PATH = PROJECT_ROOT / "M2" / "eval_harness" / "gold_examples_string.jsonl"

# Carpeta de salida
OUTPUT_DIR = PROJECT_ROOT / "M2" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Modelo (Drive):", MODEL_DIR)
print("  adapter_config existe:", (MODEL_DIR / "adapter_config.json").exists())
print("  adapter_model existe: ", (MODEL_DIR / "adapter_model.safetensors").exists())
print()
print("Gold set:", GOLD_SET_PATH)
print("  existe:", GOLD_SET_PATH.exists())
print()
print("Output:", OUTPUT_DIR)

Modelo (Drive): /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/saved_models/clinical_bert-distemist-lora
  adapter_config existe: True
  adapter_model existe:  True

Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples_string.jsonl
  existe: True

Output: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs


---
## Seccion 2 - Cargar el gold set

In [4]:
# El gold set es JSONL generado con select_rich_examples(n=75, min_entities=5, split="test")
assert GOLD_SET_PATH.exists(), (
    "Gold set no encontrado en " + str(GOLD_SET_PATH) + "\n"
    "Correr la celda de generacion de gold set en 01_dataset_preparation.ipynb primero."
)

gold_examples = []
with open(GOLD_SET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            gold_examples.append(json.loads(line))

print(f"Gold set cargado: {len(gold_examples)} ejemplos")


Gold set cargado: 59 ejemplos


In [5]:
assert len(gold_examples) > 0, "El gold set esta vacio"
assert "input" in gold_examples[0],    "Falta campo input en el gold set"
assert "esperado" in gold_examples[0], "Falta campo esperado en el gold set"
assert isinstance(gold_examples[0]["esperado"], list), (
    "esperado debe ser list[str], vino como " + str(type(gold_examples[0]["esperado"]))
)

print("Formato valido")
print("Primer ejemplo:")
print("  input[:120]:", gold_examples[0]["input"][:120], "...")
print("  esperado:", gold_examples[0]["esperado"])
print("  n_entidades:", len(gold_examples[0]["esperado"]))


Formato valido
Primer ejemplo:
  input[:120]: Varón soltero de 37 años de edad, trabajador del campo, que es enviado a nuestro Servicio desde otro hospital por no ten ...
  esperado: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis', 'carcinoma verrugoso de Buscke-Lowenstein', 'condiloma acuminado gigante de Buschke-Lowenstein', 'herida de la incisión de linfadenectomía', 'hipogonadismo hipergonadotrópico', 'infiltraba igualmente los tejidos pubianos, el escroto', 'infiltración de la grasa del tejido celular subcutáneo de la pared interna de los muslos ni de la grasa del periné', 'infiltración metastásica', 'infiltrar también la fascia del músculo aductor', 'lesiones óseas en las ramas isquio-ileo-pubianas', 'lesión', 'lesión penoescrotal', 'lesión tumoral', 'linfogranuloma venéreo', 'lúes', 'masa tumoral', 'metástasis de c

---
## Seccion 3 - Cargar el modelo fine-tuneado (BERT + LoRA)

In [6]:
# Mismo esquema BIO que en M1 (03_finetuning.ipynb, MODEL_REGISTRY)
# Redeclarado explicitamente para no depender de que adapter_config.json lo persista
label_list = ["O", "B-ENFERMEDAD", "I-ENFERMEDAD"]
id2label   = {i: l for i, l in enumerate(label_list)}
label2id   = {l: i for i, l in enumerate(label_list)}
print("id2label:", id2label)


id2label: {0: 'O', 1: 'B-ENFERMEDAD', 2: 'I-ENFERMEDAD'}


In [7]:
# Checkpoint base -- igual al MODEL_REGISTRY["clinical_BERT"] de 03_finetuning.ipynb
BASE_CHECKPOINT = "PlanTL-GOB-ES/roberta-base-biomedical-clinical-es"

base_model = AutoModelForTokenClassification.from_pretrained(
    BASE_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)
print("Modelo base cargado. Parametros totales:",
      sum(p.numel() for p in base_model.parameters()))

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  504MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo base cargado. Parametros totales: 125389827


In [8]:
import torch

!pip install --upgrade torchao

# Inyectar el adaptador LoRA entrenado en M1
# PeftModel busca adapter_config.json y adapter_model.safetensors en MODEL_DIR
model = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
model.eval()   # CRITICO: desactiva dropout para resultados deterministas

if torch.cuda.is_available():
    model = model.to("cuda")

print("Adaptador LoRA cargado")
print("Dispositivo:", next(model.parameters()).device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  504MB            

model.safetensors: downloading bytes:           |  0.00B            

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 1.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


Adaptador LoRA cargado
Dispositivo: cuda:0


In [9]:
# Cargar tokenizer desde MODEL_DIR (guardado junto al adaptador en M1 celda 38)
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
print("Tokenizer cargado. Vocab size:", tokenizer.vocab_size)


Tokenizer cargado. Vocab size: 52000


In [10]:
# Sanity check con frase de juguete antes de evaluar el gold set completo
# Esperado: diabetes->B-ENFERMEDAD, mellitus->I-ENFERMEDAD, hipertension->B-ENFERMEDAD
frase_prueba = "El paciente presenta diabetes mellitus tipo 2 y antecedentes de hipertension arterial."
tokens_prueba = frase_prueba.split()

inputs_p = tokenizer(
    tokens_prueba, is_split_into_words=True, return_tensors="pt"
).to(model.device)

with torch.no_grad():
    logits_p = model(**inputs_p).logits

pred_ids_p = logits_p.argmax(dim=-1)[0].tolist()
word_ids_p = inputs_p.word_ids(batch_index=0)

print("Sanity check -- predicciones por palabra:")
seen = set()
for idx, wid in zip(pred_ids_p, word_ids_p):
    if wid is None or wid in seen:
        continue
    print(f"  {tokens_prueba[wid]:<25} -> {id2label[idx]}")
    seen.add(wid)


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Sanity check -- predicciones por palabra:
  El                        -> O
  paciente                  -> O
  presenta                  -> O
  diabetes                  -> B-ENFERMEDAD
  mellitus                  -> I-ENFERMEDAD
  tipo                      -> I-ENFERMEDAD
  2                         -> I-ENFERMEDAD
  y                         -> O
  antecedentes              -> O
  de                        -> O
  hipertension              -> B-ENFERMEDAD
  arterial.                 -> I-ENFERMEDAD


---
## Seccion 4 - Funciones reutilizadas de M1 (sin modificar)

Estas son **exactamente** las mismas funciones definidas en `03_finetuning.ipynb`.
No se altera su logica, para que el F1 de este notebook sea directamente comparable
con el reportado al final de M1.

In [11]:
# Copiado de 03_finetuning.ipynb -- NO MODIFICAR

def strip_chunk_suffix(doc_id):
    return re.sub(r"_chunk\d+$", "", doc_id)


def bio_to_entity_set(tokens, tags):
    entities, current = [], []
    for tok, tag in zip(tokens, tags):
        if tag == "B-ENFERMEDAD":
            if current:
                entities.append(" ".join(current))
            current = [tok]
        elif tag == "I-ENFERMEDAD" and current:
            current.append(tok)
        else:
            if current:
                entities.append(" ".join(current))
            current = []
    if current:
        entities.append(" ".join(current))
    return set(e.lower().strip() for e in entities)


def aggregate_entities_by_original_doc(doc_ids, entity_sets):
    grouped = {}
    for doc_id, ents in zip(doc_ids, entity_sets):
        orig_id = strip_chunk_suffix(doc_id)
        grouped.setdefault(orig_id, set()).update(ents)
    return grouped


def micro_prf1_by_doc(true_by_doc, pred_by_doc):
    tp = fp = fn = 0
    all_doc_ids = set(true_by_doc) | set(pred_by_doc)
    for doc_id in all_doc_ids:
        true_ents = true_by_doc.get(doc_id, set())
        pred_ents = pred_by_doc.get(doc_id, set())
        tp += len(true_ents & pred_ents)
        fp += len(pred_ents - true_ents)
        fn += len(true_ents - pred_ents)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn}

print("Funciones de M1 cargadas")


Funciones de M1 cargadas


In [12]:
def predict_word_level(tokens):
    """
    Predice tag por subtoken y lo colapsa a nivel de palabra.
    Se queda con la prediccion de la primera subpalabra de cada palabra,
    igual que en M1. truncation=True usa el limite del tokenizer (512 para BERT).
    """
    inputs = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
    ).to(model.device)
    word_ids = inputs.word_ids(batch_index=0)

    with torch.no_grad():
        logits = model(**inputs).logits
    pred_ids = logits.argmax(dim=-1)[0].tolist()

    word_preds, seen = [], set()
    for idx, wid in zip(pred_ids, word_ids):
        if wid is None or wid in seen:
            continue
        word_preds.append(id2label[idx])
        seen.add(wid)
    return word_preds


---
## Seccion 5 - Calcular la metrica

In [13]:
def dimension_metrica_clasica(gold_examples):
    """
    Dimension 1 del harness M2: metrica clasica de exact-match.

    gold_examples: lista de dicts con campos: input, esperado (list[str]), criterio
    El campo esperado ya viene como lista -- no se re-parsea.

    Como el gold set no tiene doc_id, usamos el indice como id sintetico
    para que micro_prf1_by_doc funcione igual que en M1.
    """
    doc_ids, true_sets, pred_sets = [], [], []

    for i, ex in enumerate(gold_examples):
        doc_id = f"ex_{i}"
        doc_ids.append(doc_id)

        # Gold: directo desde esperado, normalizamos casing
        true_sets.append({e.lower().strip() for e in ex["esperado"]})

        # Prediccion: tokenizar -> predecir BIO -> reconstruir spans
        tokens = ex["input"].split()
        pred_tags = predict_word_level(tokens)
        pred_sets.append(bio_to_entity_set(tokens, pred_tags))

    # aggregate no-op aca: sin sufijos _chunk en ids sinteticos
    true_by_doc = aggregate_entities_by_original_doc(doc_ids, true_sets)
    pred_by_doc = aggregate_entities_by_original_doc(doc_ids, pred_sets)
    metrics     = micro_prf1_by_doc(true_by_doc, pred_by_doc)
    return metrics, true_by_doc, pred_by_doc


In [14]:
print(f"Evaluando {len(gold_examples)} ejemplos del gold set...")
metrics_exact, true_by_doc, pred_by_doc = dimension_metrica_clasica(gold_examples)

print("\n=== Dimension 1 -- Exact-match (micro-PRF1) ===")
print(json.dumps(metrics_exact, indent=2))


Evaluando 59 ejemplos del gold set...

=== Dimension 1 -- Exact-match (micro-PRF1) ===
{
  "precision": 0.3583815028901734,
  "recall": 0.3276089828269485,
  "f1": 0.34230503795721184,
  "tp": 248,
  "fp": 444,
  "fn": 509
}


In [15]:
# Chequeo de riesgo de truncamiento
n_truncados = 0
for ex in gold_examples:
    ids = tokenizer(ex["input"].split(), is_split_into_words=True, truncation=False)["input_ids"]
    if len(ids) > tokenizer.model_max_length:
        n_truncados += 1
print(f"Documentos truncados (exceden los {tokenizer.model_max_length} tokens): {n_truncados} / {len(gold_examples)}")
if n_truncados > 0:
    print("OJO: Estos truncamientos afectan negativamente el Recall (entidades no leídas se cuentan como FN falsos).")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1165 > 512). Running this sequence through the model will result in indexing errors


Documentos truncados (exceden los 512 tokens): 25 / 59
OJO: Estos truncamientos afectan negativamente el Recall (entidades no leídas se cuentan como FN falsos).


---
## Seccion 6 - Analisis de errores (evidencia para el argumento)

In [16]:
# FN y FP por ejemplo
errores_por_doc = {}
for doc_id in true_by_doc:
    fn = true_by_doc[doc_id] - pred_by_doc.get(doc_id, set())
    fp = pred_by_doc.get(doc_id, set()) - true_by_doc[doc_id]
    if fn or fp:
        errores_por_doc[doc_id] = {"fn": sorted(fn), "fp": sorted(fp)}

print(f"Ejemplos con al menos un error: {len(errores_por_doc)} / {len(true_by_doc)}")
print()
for doc_id, errs in list(errores_por_doc.items())[:5]:
    print(f"{doc_id}")
    if errs["fn"]: print("  Se le escapo (FN):", errs["fn"])
    if errs["fp"]: print("  Invento      (FP):", errs["fp"])
    print()


Ejemplos con al menos un error: 59 / 59

ex_0
  Se le escapo (FN): ['adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'afectación de las cadenas ganglionares', 'brucelosis', 'carcinoma verrugoso de buscke-lowenstein', 'condiloma acuminado gigante de buschke-lowenstein', 'herida de la incisión de linfadenectomía', 'hipogonadismo hipergonadotrópico', 'infiltraba igualmente los tejidos pubianos, el escroto', 'infiltración de la grasa del tejido celular subcutáneo de la pared interna de los muslos ni de la grasa del periné', 'infiltración metastásica', 'infiltrar también la fascia del músculo aductor', 'lesiones óseas en las ramas isquio-ileo-pubianas', 'lesión tumoral', 'linfogranuloma venéreo', 'lúes', 'masa tumoral', 'metástasis de carcinoma', 'metástasis tumorales', 'orificio fistuloso', 'pene una lesión tumoral exofítica', 'tumor', 'tumor benigno']
  Invento      (FP): ['adenopatías bilaterales en las cadenas iliacas interna y externa y en

In [17]:
# Detectar errores de boundary: prediccion es substring del gold o viceversa
# Evidencia concreta de la limitacion del exact-match

def detectar_boundary_errors(pred_ents, true_ents):
    """Pares donde una entidad es substring de la otra."""
    casos = []
    for p in pred_ents:
        for t in true_ents:
            if p != t and (p in t or t in p):
                casos.append({"predicho": p, "gold": t})
    return casos


ejemplos_boundary = []
for doc_id in true_by_doc:
    casos = detectar_boundary_errors(
        pred_by_doc.get(doc_id, set()),
        true_by_doc[doc_id]
    )
    for c in casos:
        ejemplos_boundary.append({"doc_id": doc_id, **c})

print(f"Casos de boundary/truncamiento: {len(ejemplos_boundary)}")
print()
for caso in ejemplos_boundary[:10]:
    print(f"  [{caso['doc_id']}]  predicho={caso['predicho']!r}  gold={caso['gold']!r}")


Casos de boundary/truncamiento: 510

  [ex_0]  predicho='lesión tumoral exofítica'  gold='tumor'
  [ex_0]  predicho='lesión tumoral exofítica'  gold='lesión tumoral'
  [ex_0]  predicho='lesión tumoral exofítica'  gold='pene una lesión tumoral exofítica'
  [ex_0]  predicho='lesión tumoral exofítica'  gold='lesión'
  [ex_0]  predicho='lesión'  gold='lesión penoescrotal'
  [ex_0]  predicho='lesión'  gold='lesión tumoral'
  [ex_0]  predicho='lesión'  gold='pene una lesión tumoral exofítica'
  [ex_0]  predicho='adenopatías'  gold='adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales'
  [ex_0]  predicho='adenopatías'  gold='adenopatías bilaterales ilíacas e inguinales'
  [ex_0]  predicho='lesión penoescrotal'  gold='lesión'


---
## Dimension 1 - Argumento redactado

**Que mide:** esta metrica evalua si el span de texto predicho coincide caracter a caracter
con el span anotado en el gold (normalizando casing). Es una medida directa de precision de
boundary: el modelo delimito exactamente donde empieza y termina la mencion de la enfermedad?

**Que NO mide:** es ciega a los aciertos semanticos parciales. En este corpus se encontraron
**510** casos de truncamiento como: `predicho="[ej. celda 23]"`, `gold="[ej. celda 23]"`.
El exact-match cuenta esto como fallo total, aunque el modelo ubico correctamente el inicio de
la entidad.

Esta limitacion es el gancho hacia la metrica de similitud semantica (Pau): donde el exact-match
ve dos fallos identicos (0 aciertos), la metrica semantica puede distinguir entre "no entendio
nada" y "entendio el concepto pero erro el limite del span".



---
## Seccion 7 - Exportar resultados para el harness

In [18]:
resultado_dimension1 = {
    "dimension": "metrica_clasica_exact_match",
    "rol": " ",
    "modelo": BASE_CHECKPOINT,
    "adaptador_lora": str(MODEL_DIR),
    "gold_set": str(GOLD_SET_PATH),
    "n_ejemplos_evaluados": len(true_by_doc),
    "metrics": metrics_exact,
    # Sets a listas ordenadas para serializacion determinista entre ejecuciones
    "true_by_doc": {k: sorted(v) for k, v in true_by_doc.items()},
    "pred_by_doc": {k: sorted(v) for k, v in pred_by_doc.items()},
    "casos_boundary": ejemplos_boundary,
}

output_path = OUTPUT_DIR / "resultado_dimension1.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(resultado_dimension1, f, indent=2, ensure_ascii=False)

print(f"Guardado en: {output_path}")
print()
print("Resumen final:")
print(f"  Precision : {metrics_exact['precision']:.4f}")
print(f"  Recall    : {metrics_exact['recall']:.4f}")
print(f"  F1        : {metrics_exact['f1']:.4f}")
print(f"  TP={metrics_exact['tp']}  FP={metrics_exact['fp']}  FN={metrics_exact['fn']}")

Guardado en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs/resultado_dimension1.json

Resumen final:
  Precision : 0.3584
  Recall    : 0.3276
  F1        : 0.3423
  TP=248  FP=444  FN=509


---
### Para (harness integrador)

**Archivo:** `M2/outputs/resultado_dimension1.json` (en Drive)

**Estructura:**
```json
{
  "metrics": {"precision": float, "recall": float, "f1": float, "tp": int, "fp": int, "fn": int},
  "true_by_doc": {"ex_0": [...], ...},
  "pred_by_doc": {"ex_0": [...], ...},
  "casos_boundary": [...]
}
```

**Gold set usado:** `M2/eval_harness/gold_examples_string.jsonl` -- 59 ejemplos del split test
de DisTEMIST con >= 5 entidades, generados en `01_dataset_preparation.ipynb`.

Si tu harness usa un gold set con criterio distinto, avisame para re-correr antes del scorecard final.